# 11 - Parallel KPI and dashboard refresh

Run after 03_silver_gold. It uses Spark executors for KPI scans and writes compact UI tables.

In [ ]:
import os
CATALOG=os.getenv('AIDP_CATALOG','aidp_poc'); parallelism=spark.sparkContext.defaultParallelism
spark.conf.set('spark.sql.adaptive.enabled','true'); spark.conf.set('spark.sql.shuffle.partitions',str(max(200,parallelism*4)))
print('Executor parallelism=',parallelism,'shuffle partitions=',spark.conf.get('spark.sql.shuffle.partitions'))
# Long-format KPI catalog: easy to query in dashboards without rescanning source tables.
kpis=spark.sql(f'''WITH b AS (SELECT COUNT(*) events,COUNT(DISTINCT stream_key) meters,COUNT(DISTINCT stream_partition) partitions,MAX(ingested_at) latest_ingested FROM {CATALOG}.bronze.meter_reading), s AS (SELECT COUNT(*) readings,COUNT(DISTINCT meter_id) meters,COUNT(DISTINCT interval_start_utc) intervals,ROUND(AVG(consumption_kwh),4) avg_kwh,ROUND(MAX(consumption_kwh),4) max_kwh,ROUND(AVG(voltage_v),2) avg_voltage,ROUND(AVG(power_factor),4) avg_pf,SUM(CASE WHEN quality_code='ACTUAL' THEN 1 ELSE 0 END) actual,SUM(CASE WHEN quality_code='SUSPECT' THEN 1 ELSE 0 END) suspect,SUM(CASE WHEN tamper_flag THEN 1 ELSE 0 END) tamper,SUM(CASE WHEN outage_minutes>0 THEN 1 ELSE 0 END) outage FROM {CATALOG}.silver.interval_reading), g AS (SELECT ROUND(SUM(consumption_kwh),2) kwh,ROUND(MAX(demand_kw),2) peak_kw,COUNT(DISTINCT feeder_id) feeders,COUNT(DISTINCT transformer_id) transformers,SUM(tamper_reading_count) tamper,SUM(outage_reading_count) outage FROM {CATALOG}.gold.meter_interval_usage WHERE interval_start_utc=(SELECT MAX(interval_start_utc) FROM {CATALOG}.gold.meter_interval_usage)), m AS (SELECT COUNT(*) predictions,COUNT(DISTINCT meter_id) meters,ROUND(AVG(prediction_kwh),4) avg_kwh,ROUND(MIN(prediction_kwh),4) min_kwh,ROUND(MAX(prediction_kwh),4) max_kwh,MAX(model_version) model_version,MIN(interval_start_utc) first_future_interval,MAX(interval_start_utc) last_future_interval FROM {CATALOG}.ml.meter_predictions) SELECT 'BRONZE' layer,'Events received' kpi,CAST(events AS STRING) value FROM b UNION ALL SELECT 'BRONZE','Active meters',CAST(meters AS STRING) FROM b UNION ALL SELECT 'BRONZE','Stream partitions',CAST(partitions AS STRING) FROM b UNION ALL SELECT 'BRONZE','Latest ingest',CAST(latest_ingested AS STRING) FROM b UNION ALL SELECT 'SILVER','Validated readings',CAST(readings AS STRING) FROM s UNION ALL SELECT 'SILVER','Meters covered',CAST(meters AS STRING) FROM s UNION ALL SELECT 'SILVER','15-minute intervals',CAST(intervals AS STRING) FROM s UNION ALL SELECT 'SILVER','Average kWh',CAST(avg_kwh AS STRING) FROM s UNION ALL SELECT 'SILVER','Maximum kWh',CAST(max_kwh AS STRING) FROM s UNION ALL SELECT 'SILVER','Average voltage',CAST(avg_voltage AS STRING) FROM s UNION ALL SELECT 'SILVER','Average power factor',CAST(avg_pf AS STRING) FROM s UNION ALL SELECT 'SILVER','Actual readings',CAST(actual AS STRING) FROM s UNION ALL SELECT 'SILVER','Suspect readings',CAST(suspect AS STRING) FROM s UNION ALL SELECT 'SILVER','Tamper signals',CAST(tamper AS STRING) FROM s UNION ALL SELECT 'SILVER','Outage signals',CAST(outage AS STRING) FROM s UNION ALL SELECT 'GOLD','Latest interval energy kWh',CAST(kwh AS STRING) FROM g UNION ALL SELECT 'GOLD','Peak demand kW',CAST(peak_kw AS STRING) FROM g UNION ALL SELECT 'GOLD','Feeders covered',CAST(feeders AS STRING) FROM g UNION ALL SELECT 'GOLD','Transformers covered',CAST(transformers AS STRING) FROM g UNION ALL SELECT 'GOLD','Tamper signals',CAST(tamper AS STRING) FROM g UNION ALL SELECT 'GOLD','Outage signals',CAST(outage AS STRING) FROM g UNION ALL SELECT 'ML','Predictions',CAST(predictions AS STRING) FROM m UNION ALL SELECT 'ML','Meters forecast',CAST(meters AS STRING) FROM m UNION ALL SELECT 'ML','Average forecast kWh',CAST(avg_kwh AS STRING) FROM m UNION ALL SELECT 'ML','Minimum forecast kWh',CAST(min_kwh AS STRING) FROM m UNION ALL SELECT 'ML','Maximum forecast kWh',CAST(max_kwh AS STRING) FROM m UNION ALL SELECT 'ML','Model version',CAST(model_version AS STRING) FROM m UNION ALL SELECT 'ML','First forecast interval',CAST(first_future_interval AS STRING) FROM m UNION ALL SELECT 'ML','Last forecast interval',CAST(last_future_interval AS STRING) FROM m''')
kpis.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f'{CATALOG}.gold.layer_kpis')
print('Layer KPI catalog refreshed:',kpis.count(),'KPIs')